In [38]:
###Getting data from Github

In [39]:
#Using the requests library to download the data from the evidently github repo
import requests

repo_owner = 'evidentlyai'
repo_name = 'docs'
branch_name = 'main'

zip_url = f'https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch_name}.zip'
zip_response = requests.get(zip_url)

In [40]:
#The response content in bytes
len(zip_response.content)

17545754

In [41]:
#Opening the file as a ZIP without saving to disk
import io
import zipfile

zip_archive = zipfile.ZipFile(io.BytesIO(zip_response.content))

In [42]:
filenames = zip_archive.namelist()
filenames[20:30]

['docs-main/docs/library/prompt_optimization.mdx',
 'docs-main/docs/library/report.mdx',
 'docs-main/docs/library/synthetic_data_api.mdx',
 'docs-main/docs/library/tags_metadata.mdx',
 'docs-main/docs/library/tests.mdx',
 'docs-main/docs/platform/',
 'docs-main/docs/platform/alerts.mdx',
 'docs-main/docs/platform/dashboard_add_panels.mdx',
 'docs-main/docs/platform/dashboard_add_panels_ui.mdx',
 'docs-main/docs/platform/dashboard_overview.mdx']

In [43]:
#Reading one of the files
filename = 'docs-main/docs/platform/alerts.mdx'
mdx_file = zip_archive.open(filename)
mdx_content = mdx_file.read().decode('utf-8')
print(mdx_content[:150])

---
title: 'Alerts'
description: 'How to set up alerts.'
---

<Check>
  Built-in alerting is a Pro feature available in the **Evidently Cloud** and **


In [ ]:
#Parsing metadata
import frontmatter

post = frontmatter.loads(mdx_content)
print(post.content[:100])

In [ ]:
#Removing file prefix
filename_corrected = filename.split('/', 1)[-1]
print(filename_corrected)

In [ ]:
doc = {
    'content': post.content,
    'title': post.metadata.get('title'),
    'description': post.metadata.get('description'),
    'filename': filename_corrected
}

In [ ]:
#A function to ingest/feed documentation into the system before indexing or querying it
def read_github_repository(repo_owner, repo_name, branch="main"):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/{branch}.zip"
    response = requests.get(url)
    response.raise_for_status()

    documents = []
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        for file_path in zip_ref.namelist():
            if not file_path.endswith(('.md', '.mdx')):
                continue
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                post = frontmatter.loads(content)
                doc = {
                    'content': post.content,
                    'title': post.metadata.get('title'),
                    'description': post.metadata.get('description'),
                    'filename': file_path.split('/', 1)[-1]
                }
                documents.append(doc)

    return documents

In [ ]:
repo_owner = 'evidentlyai'
repo_name = 'docs'

documents = read_github_repository(repo_owner, repo_name)

print(f"Downloaded {len(documents)} documents")

In [ ]:
!uv add gitsource

In [ ]:
#Downloading the same documentation with gitsource
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

print(f"Loaded {len(files)} documents")

In [ ]:
document = files[10]

print(document.filename)
print(document.content[:160])

In [ ]:
document.parse()

In [ ]:
#Preparing the documents for indexing
documents = [f.parse() for f in files]

In [ ]:
len(documents)

In [ ]:
##SEARCH

In [ ]:
!uv add minsearch

In [ ]:
from minsearch import Index

In [ ]:
#text_fields - fields to search with full-text search
#keyword_fields - fields for exact matching only
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(documents)

In [ ]:
#Testing if its searching within the defined fields
query = 'LLM as a Judge'
results = index.search(query, num_results=5)

In [45]:
len(results)

5

In [46]:
doc_sizes = [(doc.filename, len(doc.content)) for doc in files]
doc_sizes.sort(key=lambda x: x[1], reverse=True)

for filename, size in doc_sizes[:5]:
    print(f"{filename}: {size} characters")

metrics/all_metrics.mdx: 55085 characters
metrics/all_descriptors.mdx: 31976 characters
docs/platform/dashboard_panel_types.mdx: 31647 characters
docs/library/leftover_content.mdx: 28742 characters
metrics/customize_llm_judge.mdx: 26847 characters


In [47]:
def sliding_window(text, size=1000, step=500):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + size
        chunk = text[start:end]
        chunks.append({'start': start, 'content': chunk})

        start = end - step

        if end >= text_length:
            break

    return chunks

In [48]:
len(sliding_window(results[0]['content'], size=3000, step=2500))

39

In [49]:
document_chunks = []

for doc in documents:
    if not doc.get('content'):
        continue
    copy = doc.copy()
    content = copy.pop('content')

    chunks = sliding_window(content, size=3000, step=1500)

    for i, chunk in enumerate(chunks):
        chunk.update(copy)
        chunk['chunk_id'] = i
        document_chunks.append(chunk)

In [50]:
document_chunks[10]

{'start': 9000,
 'content': 'cation=[BinaryClassification(\n        target="target",\n        prediction_labels="prediction")],\n    categorical_columns=["target", "prediction"])\n```\n\nAvailable options and defaults:\n\n```python\n    target: str = "target"\n    prediction_labels: Optional[str] = None\n    prediction_probas: Optional[str] = "prediction" #if probabilistic classification\n    pos_label: Label = 1 #name of the positive label\n    labels: Optional[Dict[Label, str]] = None\n```\n\n### Ranking\n\n#### RecSys\n\nTo evaluate recommender systems performance, you must map the columns with:\n\n- Prediction: this could be predicted score or rank.\n- Target: relevance labels (e.g., this could be an interaction result like user click or upvote, or a true relevance label)\n\nThe **target** column can contain either:\n\n- a binary label (where `1` is a positive outcome)\n- any scores (positive values, where a higher value corresponds to a better match or a more valuable user action)

In [51]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25

In [52]:
chunk_index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
chunk_index.fit(document_chunks)

In [53]:
results = chunk_index.search(query)

In [54]:
from gitsource import chunk_documents

document_chunks = chunk_documents(documents, size=3000, step=1500)

In [55]:
document_chunks[2]

{'start': 1500,
 'content': 'ently v0.7.8">\n  ## **Evidently 0.7.8**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.8).\n</Update>\n\n<Update label="2025-06-04" description="Evidently v0.7.7">\n  ## **Evidently 0.7.7**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.7).\n</Update>\n\n<Update label="2025-05-25" description="Evidently v0.7.6">\n  ## **Evidently 0.7.6**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.6).\n</Update>\n\n<Update label="2025-05-09" description="Evidently v0.7.5">\n  ## **Evidently 0.7.5**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.5).\n</Update>\n\n<Update label="2025-05-05" description="Evidently v0.7.4">\n  ## **Evidently 0.7.4**\n\n  Full release notes on [Github](https://github.com/evidentlyai/evidently/releases/tag/v0.7.4).\n</Update>\n\n<Update label="2025-04-25